In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 88.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=90b8df5fdbe0299229bd54f7558ab4e2e67142e921a5b376ec30eb1b64d8aa69
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [2]:
def get_quantum_random_bit():
    """Generates a random bit (0 or 1) using a quantum circuit."""
    qc = QuantumCircuit(1, 1)
    qc.h(0) # Put qubit into superposition
    qc.measure(0, 0)

    # Execute on the basic simulator
    backend = BasicSimulator()
    job = transpile(qc, backend)
    result = backend.run(job).result()
    counts = result.get_counts()

    # Return the measured bit as an integer
    return int(list(counts.keys())[0])

In [10]:
# --- SETTINGS ---
N_BITS = 40
THRESHOLD = 0.15 # If error rate > 15%, an attack is reported

# --- ALICE'S PART ---
alice_bits = [get_quantum_random_bit() for _ in range(N_BITS)]
alice_bases = [get_quantum_random_bit() for _ in range(N_BITS)]

encoded_qubits = []
for i in range(N_BITS):
    qc = QuantumCircuit(1, 1)
    if alice_bits[i] == 1: qc.x(0)
    if alice_bases[i] == 1: qc.h(0)
    encoded_qubits.append(qc)

# --- EVE'S ATTACK (Intercept and Resend) ---
eve_bases = [get_quantum_random_bit() for _ in range(N_BITS)]
backend = BasicSimulator()

for i in range(N_BITS):
    qc = encoded_qubits[i]
    # Eve measures in a random basis
    if eve_bases[i] == 1: qc.h(0)
    qc.measure(0, 0)

    # Eve's measurement collapses the state.
    # To resend, she must prepare a new qubit in the state she measured.
    job = transpile(qc, backend)
    res = int(list(backend.run(job).result().get_counts().keys())[0])

    new_qc = QuantumCircuit(1, 1)
    if res == 1: new_qc.x(0)
    if eve_bases[i] == 1: new_qc.h(0)
    encoded_qubits[i] = new_qc # Bob now receives Eve's "disturbed" qubit

# --- BOB'S PART ---
bob_bases = [get_quantum_random_bit() for _ in range(N_BITS)]
bob_results = []

for i in range(N_BITS):
    qc = encoded_qubits[i]
    if bob_bases[i] == 1: qc.h(0)
    qc.measure(0, 0)
    job = transpile(qc, backend)
    bob_results.append(int(list(backend.run(job).result().get_counts().keys())[0]))

# --- SIFTING AND ERROR CHECK ---
sifted_alice = []
sifted_bob = []
for i in range(N_BITS):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

# Check a sample for errors
errors = 0
for a, b in zip(sifted_alice, sifted_bob):
    if a != b: errors += 1

error_rate = errors / len(sifted_alice) if sifted_alice else 0
print(f"Error Rate: {error_rate:.2%}")

if error_rate > THRESHOLD:
    print("!!! ATTACK DETECTED: Protocol Aborted !!!")
else:
    print("No significant disruption detected.")

Error Rate: 9.52%
No significant disruption detected.
